# 02 - Composição da base

Como a base do notebook 01 foi montada para bater com o cenário:

- **Leads**: ~280k em 180 dias. O volume varia por dia da semana (fim de semana menor) e por mês (pico em julho e novembro). Curso e canal são sorteados com pesos diferentes.
- **Canal nulo (~25%)**: a chance de vir vazio depende do canal; orgânico e e-mail chegam sem utm com mais frequência.
- **Abordagem (20%/dia, FIFO)**: dentro de cada dia os leads são ordenados pela hora de entrada e só os primeiros 20% entram no CRM, com até 3 tentativas.
- **Matrícula (~3% nos abordados)**: cada lead tem uma propensão que depende de canal, curso, modalidade e horário. A matrícula é sorteada a partir dela, e ser atendido aumenta a chance. Matrícula com data depois do fim do período fica como 0.
- **Navegação (~40% com lead_id)**: cada lead tem sessões antes e depois do cadastro. Antes do cadastro não há lead_id; na sessão do cadastro só os eventos a partir do formulário são identificados. Visitantes que nunca viraram lead completam o volume anônimo.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

FQ = "cogna.captacao"
leads = spark.table(f"{FQ}.tb_leads")
crm = spark.table(f"{FQ}.tb_abordagem_crm")
nav = spark.table(f"{FQ}.tb_navegacao")

spark.sql("USE cogna.captacao")

DataFrame[]

## 1. Volumes
Quantidade de linhas e período coberto por tabela.

In [0]:
def resumo(df, nome, col_data):
    return df.agg(F.lit(nome).alias("tabela"), F.count("*").alias("linhas"),
                  F.min(col_data).alias("inicio"), F.max(col_data).alias("fim"))

display(resumo(leads, "tb_leads", "dt_criacao")
        .unionByName(resumo(crm, "tb_abordagem_crm", "dt_tentativa_contato"))
        .unionByName(resumo(nav, "tb_navegacao", "ts_evento")))

tabela,linhas,inicio,fim
tb_leads,281526,2026-05-27T00:04:46.000Z,2026-11-22T23:59:55.000Z
tb_abordagem_crm,123898,2026-05-27T01:30:44.000Z,2026-11-22T23:59:09.000Z
tb_navegacao,6466609,2026-05-13T07:13:09.000Z,2026-11-22T23:59:59.000Z


## 2. Leads
Entrada ao longo do tempo, perfil (curso, modalidade, canal), nulos e matrícula.

In [0]:
display(leads.limit(20))

lead_id,dt_criacao,curso,modalidade,canal_origem,fl_matricula,dt_matricula
L2026052700001,2026-05-27T00:04:46.000Z,Psicologia,Presencial,google_ads,1,2026-06-01
L2026052700002,2026-05-27T00:05:43.000Z,Direito,Presencial,google_ads,0,null
L2026052700003,2026-05-27T00:18:29.000Z,Direito,Presencial,organico,0,null
L2026052700004,2026-05-27T00:21:52.000Z,Pedagogia,EAD,null,0,null
L2026052700005,2026-05-27T00:22:26.000Z,Ciências Contábeis,EAD,afiliados,0,null
L2026052700006,2026-05-27T00:25:51.000Z,Pedagogia,EAD,google_ads,0,null
L2026052700007,2026-05-27T00:28:20.000Z,Psicologia,Presencial,indicacao,0,null
L2026052700008,2026-05-27T00:33:08.000Z,Direito,Presencial,null,0,null
L2026052700009,2026-05-27T00:33:09.000Z,Análise e Desenv. de Sistemas,EAD,google_ads,0,null
L2026052700010,2026-05-27T00:40:10.000Z,Marketing,EAD,google_ads,0,null


In [0]:
# nulos por coluna
display(leads.select([F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 2).alias(c) for c in leads.columns]))

lead_id,dt_criacao,curso,modalidade,canal_origem,fl_matricula,dt_matricula
0.0,0.0,0.0,0.0,25.08,0.0,98.15


In [0]:
%sql
-- leads por dia
SELECT to_date(dt_criacao) AS dia, count(*) AS leads
FROM tb_leads
GROUP BY 1
ORDER BY 1

dia,leads
2026-05-27,1472
2026-05-28,1493
2026-05-29,1554
2026-05-30,854
2026-05-31,717
2026-06-01,1841
2026-06-02,1683
2026-06-03,1747
2026-06-04,1627
2026-06-05,1667


In [0]:
%sql
-- média por dia da semana e distribuição por hora
WITH por_dia AS (
  SELECT to_date(dt_criacao) AS dia, count(*) AS leads FROM tb_leads GROUP BY 1
)
SELECT dayofweek(dia) AS dow, date_format(dia, 'E') AS dia_semana, round(avg(leads)) AS media_leads
FROM por_dia
GROUP BY 1, 2
ORDER BY 1

dow,dia_semana,media_leads
1,Sun,892.0
2,Mon,1975.0
3,Tue,1748.0
4,Wed,1757.0
5,Thu,1770.0
6,Fri,1772.0
7,Sat,1058.0


In [0]:
%sql
SELECT hour(dt_criacao) AS hora, count(*) AS leads
FROM tb_leads
GROUP BY 1
ORDER BY 1

hora,leads
0,1971
1,2055
2,2078
3,2009
4,2012
5,1972
6,1956
8,24904
9,24566
10,24710


In [0]:
%sql
SELECT curso, modalidade, count(*) AS leads,
       round(100 * count(*) / sum(count(*)) OVER (PARTITION BY curso), 1) AS pct_no_curso,
       round(100 * sum(count(*)) OVER (PARTITION BY curso) / sum(count(*)) OVER (), 1) AS pct_curso_total
FROM tb_leads
GROUP BY 1, 2
ORDER BY pct_curso_total DESC, curso, modalidade

curso,modalidade,leads,pct_no_curso,pct_curso_total
Administração,EAD,38439,85.0,16.1
Administração,Presencial,6760,15.0,16.1
Pedagogia,EAD,31137,85.0,13.0
Pedagogia,Presencial,5499,15.0,13.0
Direito,EAD,13478,40.3,11.9
Direito,Presencial,19989,59.7,11.9
Enfermagem,EAD,11215,39.9,10.0
Enfermagem,Presencial,16917,60.1,10.0
Análise e Desenv. de Sistemas,EAD,21557,85.1,9.0
Análise e Desenv. de Sistemas,Presencial,3789,14.9,9.0


In [0]:
%sql
SELECT coalesce(canal_origem, 'vazio') AS canal_origem, count(*) AS leads,
       round(100 * count(*) / sum(count(*)) OVER (), 1) AS pct
FROM tb_leads
GROUP BY 1
ORDER BY 2 DESC

canal_origem,leads,pct
google_ads,71945,25.6
vazio,70597,25.1
meta_ads,59472,21.1
afiliados,22789,8.1
organico,22534,8.0
indicacao,20232,7.2
email_mkt,13957,5.0


In [0]:
%sql
-- matrícula por mês de entrada
SELECT date_trunc('month', dt_criacao) AS mes, count(*) AS leads, sum(fl_matricula) AS matriculas,
       round(100 * avg(fl_matricula), 2) AS pct_matricula
FROM tb_leads
GROUP BY 1
ORDER BY 1

mes,leads,matriculas,pct_matricula
2026-05-01T00:00:00.000Z,6090,115,1.89
2026-06-01T00:00:00.000Z,45430,912,2.01
2026-07-01T00:00:00.000Z,58442,1165,1.99
2026-08-01T00:00:00.000Z,45436,868,1.91
2026-09-01T00:00:00.000Z,39244,775,1.97
2026-10-01T00:00:00.000Z,46610,875,1.88
2026-11-01T00:00:00.000Z,40274,495,1.23


In [0]:
%sql
SELECT datediff(dt_matricula, to_date(dt_criacao)) AS dias_ate_matricula, count(*) AS matriculas
FROM tb_leads
WHERE fl_matricula = 1
GROUP BY 1
ORDER BY 1

dias_ate_matricula,matriculas
0,67
1,358
2,519
3,595
4,498
5,437
6,382
7,323
8,279
9,228


## 3. Abordagem CRM
Cobertura diária, tentativas, status e matrícula de abordados x não abordados.

In [0]:
display(crm.orderBy("lead_id", "dt_tentativa_contato").limit(20))

lead_id,dt_tentativa_contato,canal_contato,status_contato
L2026052700001,2026-05-27T02:43:37.000Z,WhatsApp,nao_atendeu
L2026052700001,2026-05-28T20:43:27.000Z,Telefone,nao_atendeu
L2026052700001,2026-05-30T15:26:34.000Z,WhatsApp,atendeu_em_negociacao
L2026052700002,2026-05-27T02:35:45.000Z,WhatsApp,atendeu_em_negociacao
L2026052700003,2026-05-27T05:00:38.000Z,WhatsApp,nao_atendeu
L2026052700003,2026-05-29T00:01:50.000Z,Telefone,nao_atendeu
L2026052700003,2026-05-30T19:21:15.000Z,WhatsApp,nao_atendeu
L2026052700004,2026-05-27T02:26:16.000Z,WhatsApp,atendeu_sem_interesse
L2026052700005,2026-05-27T05:18:32.000Z,WhatsApp,nao_atendeu
L2026052700005,2026-05-28T02:25:59.000Z,WhatsApp,nao_atendeu


In [0]:
# uma linha por lead abordado: 1ª tentativa, última tentativa e status final
w = Window.partitionBy("lead_id").orderBy("dt_tentativa_contato")
crm_lead = (crm
    .withColumn("n", F.row_number().over(w))
    .withColumn("qtd_tentativas", F.count("*").over(Window.partitionBy("lead_id")))
    .withColumn("dt_primeira", F.min("dt_tentativa_contato").over(Window.partitionBy("lead_id")))
    .filter(F.col("n") == F.col("qtd_tentativas"))
    .select("lead_id", "qtd_tentativas", "dt_primeira", F.col("status_contato").alias("status_final")))

leads_crm = (leads.join(crm_lead, "lead_id", "left")
    .withColumn("abordado", F.col("qtd_tentativas").isNotNull().cast("int")))
leads_crm.createOrReplaceTempView("vw_leads_crm")

In [0]:
%sql
-- % abordado por dia
SELECT to_date(dt_criacao) AS dia, count(*) AS leads, sum(abordado) AS abordados,
       round(100 * avg(abordado), 1) AS pct_abordados
FROM vw_leads_crm
GROUP BY 1
ORDER BY 1

dia,leads,abordados,pct_abordados
2026-05-27,1472,295,20.0
2026-05-28,1493,299,20.0
2026-05-29,1554,311,20.0
2026-05-30,854,171,20.0
2026-05-31,717,144,20.1
2026-06-01,1841,369,20.0
2026-06-02,1683,337,20.0
2026-06-03,1747,350,20.0
2026-06-04,1627,326,20.0
2026-06-05,1667,334,20.0


In [0]:
%sql
-- % abordado por hora de entrada
SELECT hour(dt_criacao) AS hora, count(*) AS leads, round(100 * avg(abordado), 1) AS pct_abordados
FROM vw_leads_crm
GROUP BY 1
ORDER BY 1

hora,leads,pct_abordados
0,1971,100.0
1,2055,100.0
2,2078,100.0
3,2009,100.0
4,2012,100.0
5,1972,100.0
6,1956,100.0
8,24904,100.0
9,24566,70.9
10,24710,0.0


In [0]:
%sql
SELECT qtd_tentativas, status_final, count(*) AS leads
FROM vw_leads_crm
WHERE abordado = 1
GROUP BY 1, 2
ORDER BY 1, 3 DESC

qtd_tentativas,status_final,leads
1,atendeu_sem_interesse,6526
1,atendeu_em_negociacao,5932
1,numero_invalido,5000
1,atendeu_agendou_visita,1333
1,nao_atendeu,198
2,atendeu_sem_interesse,3368
2,atendeu_em_negociacao,2905
2,atendeu_agendou_visita,724
2,nao_atendeu,276
3,nao_atendeu,27941


In [0]:
%sql
SELECT canal_contato, status_contato, count(*) AS tentativas,
       round(100 * count(*) / sum(count(*)) OVER (PARTITION BY canal_contato), 1) AS pct_no_canal
FROM tb_abordagem_crm
GROUP BY 1, 2
ORDER BY 1, 3 DESC

canal_contato,status_contato,tentativas,pct_no_canal
Telefone,nao_atendeu,38181,77.2
Telefone,atendeu_sem_interesse,4416,8.9
Telefone,atendeu_em_negociacao,3920,7.9
Telefone,numero_invalido,1993,4.0
Telefone,atendeu_agendou_visita,941,1.9
WhatsApp,nao_atendeu,57749,77.6
WhatsApp,atendeu_sem_interesse,6544,8.8
WhatsApp,atendeu_em_negociacao,5809,7.8
WhatsApp,numero_invalido,3007,4.0
WhatsApp,atendeu_agendou_visita,1338,1.8


In [0]:
%sql
-- horas entre a criação do lead e a 1ª tentativa
SELECT floor((unix_timestamp(dt_primeira) - unix_timestamp(dt_criacao)) / 3600) AS horas, count(*) AS leads
FROM vw_leads_crm
WHERE abordado = 1
GROUP BY 1
ORDER BY 1

horas,leads
0,3320
1,21135
2,14372
3,7473
4,4031
5,2269
6,1337
7,807
8,528
9,348


In [0]:
%sql
SELECT CASE WHEN abordado = 1 THEN 'abordado' ELSE 'nao_abordado' END AS grupo,
       count(*) AS leads, sum(fl_matricula) AS matriculas, round(100 * avg(fl_matricula), 2) AS pct_matricula
FROM vw_leads_crm
GROUP BY 1

grupo,leads,matriculas,pct_matricula
nao_abordado,225143,3642,1.62
abordado,56383,1563,2.77


## 4. Navegação
Identificação dos eventos, tipos de sessão, páginas e tempo de permanência.

In [0]:
display(nav.filter("lead_id IS NOT NULL").limit(20))

session_id,lead_id,ts_evento,pagina,tempo_permanencia_seg
fae9cc6dc7361269deaf,L2026052700001,2026-05-27T00:04:46.000Z,cadastro_lead,52
fae9cc6dc7361269deaf,L2026052700001,2026-05-27T00:06:01.000Z,lista_cursos,12
fae9cc6dc7361269deaf,L2026052700001,2026-05-27T00:07:16.000Z,pagina_curso,80
fae9cc6dc7361269deaf,L2026052700001,2026-05-27T00:08:31.000Z,ead_como_funciona,72
fae9cc6dc7361269deaf,L2026052700001,2026-05-27T00:09:46.000Z,home,75
fae9cc6dc7361269deaf,L2026052700001,2026-05-27T00:11:01.000Z,lista_cursos,5
fae9cc6dc7361269deaf,L2026052700001,2026-05-27T00:12:16.000Z,lista_cursos,31
bdc1d63f792764552a91,L2026052700001,2026-06-13T12:01:45.000Z,pagina_curso,28
bdc1d63f792764552a91,L2026052700001,2026-06-13T12:03:25.000Z,home,48
bdc1d63f792764552a91,L2026052700001,2026-06-13T12:03:37.000Z,blog,28


In [0]:
# resumo por sessão (base para as próximas células)
sessoes = (nav.groupBy("session_id")
    .agg(F.count("*").alias("eventos"),
         F.count("lead_id").alias("eventos_ident"),
         F.first("lead_id", ignorenulls=True).alias("lead_id"))
    .withColumn("tipo_sessao", F.when(F.col("eventos_ident") == 0, "anonima")
                                .when(F.col("eventos_ident") == F.col("eventos"), "identificada")
                                .otherwise("mista")))

In [0]:
tot = sessoes.agg(F.sum("eventos").alias("eventos"), F.sum("eventos_ident").alias("eventos_ident"),
                  F.count("*").alias("sessoes"), F.countDistinct("lead_id").alias("leads_com_nav"))
qtd_leads = leads.count()
display(tot.select("*",
    F.round(100 * F.col("eventos_ident") / F.col("eventos"), 1).alias("pct_eventos_ident"),
    F.round(100 * F.col("leads_com_nav") / F.lit(qtd_leads), 1).alias("pct_leads_com_nav")))

eventos,eventos_ident,sessoes,leads_com_nav,pct_eventos_ident,pct_leads_com_nav
6466609,2557532,1345011,281526,39.5,100.0


In [0]:
display(sessoes.groupBy("tipo_sessao")
        .agg(F.count("*").alias("sessoes"), F.sum("eventos").alias("eventos"),
             F.sum(F.col("eventos") - F.col("eventos_ident")).alias("eventos_sem_lead"))
        .orderBy(F.desc("sessoes")))

tipo_sessao,sessoes,eventos,eventos_sem_lead
anonima,715652,3133040,3133040
identificada,347833,1786396,0
mista,281526,1547173,776037


In [0]:
# exemplo de sessão mista: antes do cadastro sem lead_id, depois com
exemplo = sessoes.filter("tipo_sessao = 'mista' AND eventos >= 5").select("session_id").limit(1)
display(nav.join(exemplo, "session_id").orderBy("ts_evento"))

session_id,lead_id,ts_evento,pagina,tempo_permanencia_seg
d39cf72b926b5025afd5,null,2026-05-27T10:50:24.000Z,blog,14
d39cf72b926b5025afd5,null,2026-05-27T10:51:39.000Z,blog,5
d39cf72b926b5025afd5,null,2026-05-27T10:52:54.000Z,lista_cursos,35
d39cf72b926b5025afd5,null,2026-05-27T10:54:09.000Z,lista_cursos,194
d39cf72b926b5025afd5,null,2026-05-27T10:55:24.000Z,pagina_curso,15
d39cf72b926b5025afd5,L2026052700484,2026-05-27T10:56:39.000Z,cadastro_lead,32
d39cf72b926b5025afd5,L2026052700484,2026-05-27T10:57:54.000Z,pagina_curso,49
d39cf72b926b5025afd5,L2026052700484,2026-05-27T10:59:09.000Z,blog,74
d39cf72b926b5025afd5,L2026052700484,2026-05-27T11:00:24.000Z,lista_cursos,101


In [0]:
display(sessoes.groupBy(F.least(F.col("eventos"), F.lit(30)).alias("eventos_na_sessao"))
        .count().orderBy("eventos_na_sessao"))

eventos_na_sessao,count
1,243328
2,251201
3,195478
4,145023
5,110690
6,85830
7,65208
8,51374
9,40127
10,31129


In [0]:
# páginas: volume, participação e tempo de permanência
paginas = (nav.groupBy("pagina")
    .agg(F.count("*").alias("eventos"),
         F.percentile_approx("tempo_permanencia_seg", 0.5).alias("mediana_seg"),
         F.percentile_approx("tempo_permanencia_seg", 0.9).alias("p90_seg")))
display(paginas.crossJoin(paginas.agg(F.sum("eventos").alias("total")))
        .withColumn("pct", F.round(100 * F.col("eventos") / F.col("total"), 1))
        .drop("total")
        .orderBy(F.desc("eventos")))

pagina,eventos,mediana_seg,p90_seg,pct
pagina_curso,1842478,33,108,28.5
lista_cursos,1589675,33,108,24.6
home,1085650,33,107,16.8
blog,1022580,33,107,15.8
ead_como_funciona,422378,34,110,6.5
cadastro_lead,281526,31,102,4.4
preco_mensalidade,165741,58,188,2.6
bolsas_descontos,36614,59,191,0.6
vestibular_inscricao,9433,60,193,0.1
matricula_pagamento,5205,61,195,0.1


In [0]:
display(nav.groupBy(F.least(F.floor(F.col("tempo_permanencia_seg") / 15) * 15, F.lit(600)).alias("faixa_seg"))
        .count().orderBy("faixa_seg"))

faixa_seg,count
0,1232629
15,1675264
30,1115599
45,715558
60,470348
75,320168
90,223404
105,160751
120,117088
135,87315


## 5. Integridade
Chaves órfãs, duplicidade e datas inconsistentes.

In [0]:
ids = leads.select("lead_id")
checks = [
    ("crm sem lead", crm.select("lead_id").join(ids, "lead_id", "left_anti").count()),
    ("navegacao sem lead", nav.filter("lead_id IS NOT NULL").select("lead_id").distinct().join(ids, "lead_id", "left_anti").count()),
    ("lead_id duplicado", qtd_leads - ids.distinct().count()),
    ("matricula antes da criacao", leads.filter(F.col("dt_matricula") < F.to_date("dt_criacao")).count()),
]
display(spark.createDataFrame(checks, "checagem string, ocorrencias long"))

checagem,ocorrencias
crm sem lead,0
navegacao sem lead,0
lead_id duplicado,0
matricula antes da criacao,0
